In [ ]:
from VAE.VQ_VAE import VQVAE
from Pixel_CNN.pixel_cnn import PixelCNN
from Pixel_CNN.pixel_cnn_utils import train, optimizer
import torch
from datasets.latent_dataset import get_dataloader
from utils.image_construction import reconstruct_from_indices, save_image

import json

### Hyperparams

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("models/pixelcnn_v1_hyperparams.json", "r") as f:
    hyperparams = json.load(f)

input_dim = hyperparams["input_dim"]
output_dim = hyperparams["output_dim"]
num_res = hyperparams["residual_blocks"]
filters = hyperparams["filters"]

epochs = hyperparams["epochs"]
batch_size = hyperparams["batch_size"]

### Load Dataset

In [3]:
image_folder = 'latent_images'

train_loader = get_dataloader(image_folder, batch_size, n=200000)

Found 200000 latent indices


### Create Model

In [4]:
model = PixelCNN(input_dim, output_dim, num_res, filters).to(device)

### VQ-VAE for Testing

In [5]:
vae = VQVAE(64, 1024, 1)

vae.load_state_dict(torch.load('models/vae_checkpoint_v1.pth', weights_only=True))
vae.to(device);

In [6]:
training_state_path = 'state.json'

try:
    with open(training_state_path, 'r') as file:
        training_state_data = json.load(file)
    model.load_state_dict(torch.load(f"models/pixelcnn_checkpoint_{training_state_data['last_epoch']}.pth", weights_only=True))
except:
    training_state_data = {
        'last_epoch': 0,
        'lr': 3e-4,
        'loss': [],
    }

for epoch in range(training_state_data['last_epoch'] + 1, epochs + 1):
    lr = training_state_data['lr']
    opt = optimizer(model, learning_rate=lr)
    loss = train(model, train_loader, opt, device)
    
    print(f"\nEpoch [{epoch}/{epochs}]")
    print(f"Loss: {loss:.4f}")
    
    for letter in ['a', 'b', 'c', 'd', 'e']:
        z = model.sample((16, 16), device)
        z = reconstruct_from_indices(z, vae, device)
        save_image(f'output/epoch{epoch}{letter}.png', z)
    
    if epoch % 15 == 0:
        print('Reducing LR')
        training_state_data['lr'] /= 2
    
    training_state_data['last_epoch'] = epoch
    training_state_data['loss'].append(loss)
    
    torch.save(model.state_dict(), f"models/pixelcnn_checkpoint_{epoch}.pth")
    with open(training_state_path, 'w') as state_file:
        json.dump(training_state_data, state_file)

100%|██████████| 1563/1563 [08:25<00:00,  3.09it/s]



Epoch [16/100]
Loss: 5.8911


100%|██████████| 1563/1563 [04:25<00:00,  5.89it/s]



Epoch [17/100]
Loss: 5.8884


100%|██████████| 1563/1563 [08:33<00:00,  3.04it/s]


...


Epoch [98/100]
Loss: 5.8456


100%|██████████| 1563/1563 [03:15<00:00,  8.00it/s]



Epoch [99/100]
Loss: 5.8455


100%|██████████| 1563/1563 [07:38<00:00,  3.41it/s]



Epoch [100/100]
Loss: 5.8455
